# Session 6 · Error and the Cost Function

**Machine Learning Foundations · Sanketana School of Code**

Last session ended with an argument we couldn't win: your line and sklearn's line disagreed, and we couldn't say whose was better without squinting. Today we settle it — by inventing a **number for how wrong a line is.**

By the end of this notebook you will be able to:

- measure one student's **miss** (how far off the line is for them)
- explain why we **square** the misses instead of just adding them
- compute the **cost** of a line — one number — and use it to pick the better line
- draw the **bowl** and say what `model.fit()` actually does

## Warm-up · Last session's homework

Your coach will walk through Session 5's housing line (about 10 minutes). You each got a slope, an intercept, and a prediction.

And you were left with a nagging question: your by-eye line and sklearn's line came out **different** — so *whose line is actually better, and how would we prove it?* That's today.

## From “which line?” to “how wrong is this line?”

We can't rank lines by looking. So we'll give every line a score for **badness** — its **cost**. Then "best line" just means "smallest cost," and the argument is over.

We use the same data and the same single feature as Session 5, so today's ideas attach to the exact line you drew.

## Step 1 · One student's miss

Fit sklearn's line (you know this move), then look at a few students. For each, the **miss** is how far their real score is from the line:

```
miss = actual − predicted
```

Above the line → positive miss. Below the line → negative miss.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

students = pd.read_csv("../../../datasets/anchor/student_habits.csv")
feature, label = "study_hours_per_week", "test_score"
x = students[feature].values
y = students[label].values

# sklearn's best-fit line (single feature).
model = LinearRegression().fit(students[[feature]], y)
sk_slope, sk_intercept = model.coef_[0], model.intercept_
print(f"sklearn line: score = {sk_slope:.2f} × study_hours + {sk_intercept:.2f}")

In [ ]:
# Misses for the first 6 students, under sklearn's line.
look = students.head(6).copy()
look["predicted"] = sk_slope * look[feature] + sk_intercept
look["miss"] = look[label] - look["predicted"]
look[[feature, label, "predicted", "miss"]].round(1)

## Step 2 · Why we can't just add the misses

Obvious idea: add up everyone's miss, call it the badness. It fails — in a sneaky way. Watch a tiny 2-student example: one student `+10` above the line, one `−10` below it.

In [ ]:
toy = np.array([10, -10])   # two misses: one above, one below

print("raw sum of misses:    ", toy.sum(), "  ← looks perfect, but both were off by 10!")
print("sum of SQUARED misses:", (toy**2).sum(), "  ← honestly says the line missed both")

The `+10` and `−10` **cancelled** to zero. And this isn't rare — sklearn's own best line balances the dots above and below, so its raw misses also add up to almost exactly zero:

In [ ]:
all_misses = y - (sk_slope * x + sk_intercept)
print("sklearn line, raw sum of misses:    ", round(all_misses.sum(), 4))
print("sklearn line, sum of SQUARED misses:", round((all_misses**2).sum(), 1))
print("\nRaw sum ≈ 0 for a good line AND for many bad ones → useless as a score.")

So we **square** every miss before adding. Squaring:

1. removes the sign (a `−10` and a `+10` both become `+100`) — nothing cancels;
2. punishes big misses harder than small ones (a miss of 10 costs 100; a miss of 2 costs just 4).

## Step 3 · The cost of a line

Here's the number we wanted. The **cost** of a line is the **average squared miss** across all students — one number for the whole line:

```
cost = average of (actual − predicted)²
```

The helper below computes it for any slope and intercept. Lower cost = better line.

In [ ]:
def cost(slope, intercept):
    """Average squared miss of the line (slope, intercept) over all students."""
    predicted = slope * x + intercept
    misses = y - predicted
    return np.mean(misses**2)

# ✏️ TODO: put YOUR by-eye line from Session 5 here (or keep this guess).
my_slope, my_intercept = 2.0, 40.0

print(f"cost of MY line     ({my_slope}, {my_intercept}): {cost(my_slope, my_intercept):.1f}")
print(f"cost of SKLEARN line ({sk_slope:.2f}, {sk_intercept:.2f}): {cost(sk_slope, sk_intercept):.1f}")

### ✏️ Settle the contest

1. Which line has the lower cost — yours or sklearn's? So which one is better, *by the number*?
2. Last session we could only argue about this by looking at the plot. In one sentence, why is having a **cost number** better than eyeballing?

*Your answers:*

1. 
2. 

## Step 4 · Draw the bowl

Now the picture that ties it together. Forget the students for a second and ask: **as I change the slope, what happens to the cost?**

We'll hold the intercept fixed at sklearn's value, try many slopes, and plot each slope's cost. *Every point on this curve is one whole line.*

In [ ]:
# Try many slopes; for each, compute the cost (intercept fixed at sklearn's).
slopes = np.linspace(0, 6, 61)
costs = [cost(s, sk_intercept) for s in slopes]

plt.figure(figsize=(7, 5))
plt.plot(slopes, costs, color="purple")
plt.xlabel("slope of the line (points per study hour)")
plt.ylabel("cost (average squared miss)")
plt.title("The bowl: every slope has a cost")
plt.show()

A **valley**. Slopes that are too flat sit high on the left wall; too steep, high on the right; the best slope sits at the very **bottom**. Remember: each dot on this curve is an entire candidate line.

## Step 5 · Find the floor — and meet `.fit()`

The lowest point of the bowl is the best line. Let's find it and check it against what `model.fit()` gave us.

In [ ]:
best_i = int(np.argmin(costs))
best_slope = slopes[best_i]

print(f"bottom of the bowl is at slope ≈ {best_slope:.2f}")
print(f"sklearn .fit() chose slope   = {sk_slope:.2f}")

plt.figure(figsize=(7, 5))
plt.plot(slopes, costs, color="purple")
plt.scatter([best_slope], [costs[best_i]], color="red", zorder=5, label="bottom (best line)")
plt.axvline(sk_slope, color="green", linestyle="--", label="sklearn .fit() slope")
plt.xlabel("slope"); plt.ylabel("cost")
plt.title("The floor of the bowl is what .fit() finds")
plt.legend(); plt.show()

They land in the same place. So here is the demystified answer to "how does the model learn the line?":

> **`model.fit()` slides to the bottom of the bowl** — it finds the slope and intercept with the smallest cost.

It doesn't try every line like we just did (we only did that to *see* the bowl); it feels which way is downhill and steps that way until it can't go lower.

> **Tempted to push the cost to zero with a wiggly line?** Hold that exact thought — it's the trap we study in **Session 16**. Today: straight lines, and "best" means the bottom of this bowl.

### ✏️ Say it in your own words

In one sentence, no maths: what is `model.fit()` doing?

*Your answer:* 

### ✏️ Stretch — a worse intercept lifts the whole bowl

The bowl above used sklearn's *good* intercept. What if the intercept were wrong? Re-draw the bowl with a bad intercept and compare.

In [ ]:
bad_intercept = sk_intercept + 20   # push the whole line too high
costs_bad = [cost(s, bad_intercept) for s in slopes]

plt.figure(figsize=(7, 5))
plt.plot(slopes, costs, color="purple", label=f"good intercept ({sk_intercept:.0f})")
plt.plot(slopes, costs_bad, color="orange", label=f"bad intercept ({bad_intercept:.0f})")
plt.xlabel("slope"); plt.ylabel("cost")
plt.title("A worse intercept raises the whole valley")
plt.legend(); plt.show()

print("The real cost landscape is a bowl over BOTH slope and intercept.")
print("We only sliced it at one intercept to draw it in 2-D.")

## What we learned

✏️ Three quick reflections — one line each:

1. Why we square the misses instead of adding them raw:
2. What one dot on the bowl represents:
3. What sits at the bottom of the bowl:

---

**You built the missing number.** Every line has a **cost** — the average squared miss — and the best line is the one with the smallest cost, sitting at the bottom of the **bowl**. `model.fit()` just slides to that floor. The Session 5 argument is settled, with a number.

**Next (Session 7):** one feature gave us one slope and one bowl. Real problems have many ingredients at once — study hours *and* sleep *and* attendance. We go from one slope to several, and ask which ingredient matters most.